# Colab 10 — ¿Cuánto tarda en morirse esta oscilación, y cómo sé que mi ajuste es correcto?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 10 — 14/10

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/10_Ajuste_no_lineal_y_por_que_R2_miente.ipynb)

Es el punto de llegada de todo el cuatrimestre. Cinco parámetros libres, un modelo genuinamente no lineal, y la pregunta de siempre —¿está bien mi ajuste?— que ahora ya no se puede contestar con un solo número.

**Al terminar vas a poder:** ajustar modelos no lineales eligiendo semillas con criterio físico, leer una matriz de correlación de parámetros, y explicar por qué $R^2$ no sirve acá.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. El modelo

$$x(t) = A\,e^{-t/\tau}\cos(\omega t + \phi) + x_0$$

Cinco parámetros. Y a diferencia de todo lo que ajustamos hasta ahora, éste
es **no lineal en los parámetros**: no hay fórmula cerrada, la rutina busca
el mínimo iterativamente, y **puede no encontrarlo**.

In [ ]:
generador = np.random.default_rng(2026)

A_r, tau_r, w_r, phi_r, x0_r = 0.0620, 4.35, 7.912, 0.42, 0.0011

t = np.arange(0, 14, 0.01)
sigma = 0.0009
x = A_r*np.exp(-t/tau_r)*np.cos(w_r*t + phi_r) + x0_r
x = x + generador.normal(0, sigma, size=len(t))
sx = np.full(len(t), sigma)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, x, "-", lw=0.7)
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Posición (m)")
plt.show()

### 2. Las semillas salen de la física, no de probar

`curve_fit` necesita un punto de partida `p0`. Si no se lo das, arranca en
todo unos, y con un coseno adentro eso **casi siempre falla**: el ajuste se
queda en un mínimo local con una frecuencia equivocada.

La buena noticia es que las cinco semillas se leen del gráfico:

- $A$: la amplitud inicial.
- $x_0$: el valor alrededor del cual oscila (el promedio de la cola).
- $\omega$: contá picos. $\omega = 2\pi \times (\text{picos}/\text{tiempo})$.
- $\tau$: el tiempo en que la envolvente cae a $1/e \approx 0{,}37$ de la
  amplitud inicial.
- $\phi$: cero sirve; el ajuste lo acomoda.

In [ ]:
from scipy.signal import find_peaks


def amortiguado(t, A, tau, w, phi, x0):
    return A*np.exp(-t/tau)*np.cos(w*t + phi) + x0


picos, _ = find_peaks(x, distance=50)
t_picos, x_picos = t[picos], x[picos]

A0 = x_picos[0] - x[-200:].mean()
x00 = x[-200:].mean()
w0 = 2*np.pi*len(t_picos)/(t_picos[-1] - t_picos[0]) * (len(t_picos)-1)/len(t_picos)
por_debajo = np.where(x_picos - x00 < A0/np.e)[0]
tau0 = t_picos[por_debajo[0]] if len(por_debajo) else 5.0

p0 = [A0, tau0, w0, 0.0, x00]
print("semillas leídas del gráfico:")
for nombre, valor in zip(["A", "tau", "omega", "phi", "x0"], p0):
    print(f"  {nombre:<7} {valor:.4f}")

In [ ]:
print("Con semillas de la física:")
p, e, cov = lab.ajustar(amortiguado, t, x, yerr=sx, p0=p0,
                        nombres=["A (m)", "tau (s)", "omega (rad/s)",
                                 "phi (rad)", "x0 (m)"])

In [ ]:
# Y ahora sin semillas, para que veas el modo de falla.
from scipy.optimize import curve_fit

try:
    p_mal, _ = curve_fit(amortiguado, t, x, sigma=sx, absolute_sigma=True,
                         maxfev=20000)
    print("convergió a:", np.round(p_mal, 4))
    print(f"omega verdadero = {w_r:.3f} rad/s")
    print("Convergió, sí. A otra cosa.")
except RuntimeError as err:
    print("no convergió:", err)

Ése es el modo de falla peligroso del ajuste no lineal: **no tira error**.
Devuelve parámetros perfectamente formateados que no describen nada. La
única defensa es mirar la curva superpuesta a los datos, siempre.

### 3. La matriz de correlación de los parámetros

Reportar cinco parámetros con sus cinco errores por separado es una
descripción incompleta, porque los parámetros **no son independientes**.

In [ ]:
nombres = ["A", "tau", "omega", "phi", "x0"]
M = lab.matriz_correlacion(cov)

print("        " + "".join(f"{n:>9}" for n in nombres))
for i, n in enumerate(nombres):
    print(f"{n:<8}" + "".join(f"{M[i,j]:>9.2f}" for j in range(len(nombres))))

$A$ y $\tau$ salen fuertemente correlacionados, y es fácil ver por qué:
subir la amplitud inicial y acortar el tiempo de decaimiento producen
envolventes casi iguales en la zona donde hay datos. El ajuste no puede
separarlos bien y lo dice.

Esto tiene una consecuencia práctica inmediata: si después usás $A$ y $\tau$
juntos para calcular otra cosa —por ejemplo la energía disipada— propagar
tratándolos como independientes está **mal**, y el error te va a dar
inflado o desinflado según el signo de la correlación.

Es exactamente lo que pasa al ajustar un espectro de impedancia con un
circuito equivalente: los parámetros de un mismo elemento salen correlacionados
y el error nominal de cada uno por separado no significa gran cosa.

### 4. Por qué R² miente

Ajustamos los mismos datos con dos modelos **incorrectos**, uno grosero y
uno sutil, y comparamos $R^2$ contra $\chi^2_\nu$.

- El grosero **ignora el amortiguamiento**: una sinusoide pura.
- El sutil tiene amortiguamiento, pero de la forma equivocada: la envolvente
  decae como $1/(1+t/\tau)$ en lugar de exponencialmente. Tiene los mismos
  cinco parámetros que el correcto y es indistinguible a ojo.

In [ ]:
def sin_amortiguar(t, A, w, phi, x0):
    return A*np.cos(w*t + phi) + x0


def decaimiento_algebraico(t, A, tau, w, phi, x0):
    return A*np.cos(w*t + phi)/(1 + t/tau) + x0


p_m1, _, _ = lab.ajustar(sin_amortiguar, t, x, yerr=sx,
                         p0=[A0, w0, 0.0, x00], verbose=False)
p_m2, _, _ = lab.ajustar(decaimiento_algebraico, t, x, yerr=sx, p0=p0,
                         verbose=False)

filas = [("correcto (exponencial)", amortiguado, p, 5),
         ("incorrecto sutil (algebraico)", decaimiento_algebraico, p_m2, 5),
         ("incorrecto grosero (sin amortiguar)", sin_amortiguar, p_m1, 4)]

print(f"{'modelo':<36}{'R²':>9}{'χ²_ν':>10}{'p-valor':>12}")
for nombre, f, par, k in filas:
    r2 = lab.R2(x, f(t, *par))
    c2, pv = lab.chi2_reducido(x, f(t, *par), sx, k, verbose=False)
    print(f"{nombre:<36}{r2:9.4f}{c2:10.2f}{pv:12.3g}")

c2_bueno, p_bueno = lab.chi2_reducido(x, amortiguado(t, *p), sx, 5,
                                      verbose=False)
c2_malo, p_pmalo = lab.chi2_reducido(x, decaimiento_algebraico(t, *p_m2), sx,
                                     5, verbose=False)

Ahí está el punto de toda la clase, y está en la fila del medio.

Con el modelo **groseramente** equivocado, $R^2$ se derrumba: hasta ahí, todo
bien, cualquier indicador lo detecta. Pero con el modelo **sutilmente**
equivocado —el que en la práctica uno realmente corre el riesgo de usar—
$R^2$ da 0,97 contra 0,997 del correcto. Nadie descarta un modelo por eso. El
$\chi^2_\nu$, en cambio, se va a más de diez y el p-valor es
indistinguible de cero.

La razón de fondo: $R^2$ compara tu modelo contra el modelo trivial "todo
vale el promedio", y **no usa las barras de error para nada**. Es un
cociente de varianzas, no una prueba estadística. Depende además del rango
muestreado: con el mismo modelo y los mismos errores, medir en un rango más
ancho sube $R^2$ sin que nada haya mejorado.

La referencia formal es Spiess y Neumeyer, *BMC Pharmacology* 10:6 (2010):
con datos simulados muestran que $R^2$ puede ser alto para un modelo
francamente incorrecto y bajo para el correcto, según el ruido y el rango.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                         gridspec_kw={"height_ratios": [1, 1]})
for ax, modelo, par, titulo, c2 in [
        (axes[0], amortiguado, p, "modelo correcto", c2_bueno),
        (axes[1], decaimiento_algebraico, p_m2, "modelo incorrecto sutil",
         c2_malo)]:
    ax.plot(t, (x - modelo(t, *par))/sx, ".", ms=2)
    ax.axhline(0, color="crimson", lw=1)
    ax.set_ylabel("residuo / σ")
    ax.set_title(f"{titulo} — χ²_ν = {c2:.1f}")
axes[1].set_xlabel("Tiempo (s)")
plt.show()

Y los residuos son todavía más elocuentes que cualquier número: los del
modelo incorrecto **tienen la forma de lo que falta**, una modulación que
crece y decrece según dónde la envolvente equivocada se separa de la buena.
Cuando en un panel de residuos ves estructura, no hay estadístico que
discutir.

### 5. Linealizar contra ajustar: cuánto cuesta

Ahora cuantificamos lo que anunciamos en la Clase 5. Sacamos $\tau$ de dos
maneras: ajustando una recta a $\ln$(amplitud de los picos) —sin ponderar,
que es lo que se hace habitualmente— y con el ajuste no lineal completo.

In [ ]:
def recta(x, a, b):
    return a*x + b


amplitudes = x_picos - p[4]
buenos = amplitudes > 5*sigma

ln_amp = np.log(amplitudes[buenos])
t_amp = t_picos[buenos]

# Ajuste sin ponderar en el espacio logarítmico (lo habitual, y sesgado).
p_lin, e_lin, _ = lab.ajustar(recta, t_amp, ln_amp, verbose=False)
tau_lin, stau_lin = -1/p_lin[0], e_lin[0]/p_lin[0]**2

# Ajuste ponderado, propagando el error al espacio logarítmico.
s_ln = sigma/amplitudes[buenos]
p_pond, e_pond, _ = lab.ajustar(recta, t_amp, ln_amp, yerr=s_ln, verbose=False)
tau_pond, stau_pond = -1/p_pond[0], e_pond[0]/p_pond[0]**2

print(f"tau linealizado sin pesos : {lab.formatear(tau_lin, stau_lin, 's')}")
print(f"tau linealizado con pesos : {lab.formatear(tau_pond, stau_pond, 's')}")
print(f"tau del ajuste no lineal  : {lab.formatear(p[1], e[1], 's')}")
print(f"tau verdadero (simulado)  : {tau_r} s")
print()
lab.compatibilidad(tau_lin, stau_lin, p[1], e[1],
                   etiquetas=("linealizado sin pesos", "no lineal directo"))

La diferencia entre las dos determinaciones suele ser **mayor que sus
propios errores**, y el mecanismo es el que ya vimos: al tomar logaritmo, los
picos chicos —los más ruidosos en términos relativos— quedan con barras
enormes, y ajustar sin ponderar les da la misma influencia que a los grandes.

Conclusión operativa: linealizar es legítimo para **estimar semillas** y para
mirar los datos, pero el resultado que se informa sale del ajuste no lineal
sobre los datos originales. Si por alguna razón hay que ajustar en el espacio
transformado, se propaga la incerteza y se pondera.

### 6. La figura final

In [ ]:
fig, axes = lab.grafico_con_residuos(
    t, x, amortiguado, p, yerr=sx, normalizar_residuos=True,
    xlabel="Tiempo (s)", ylabel="Posición (m)",
    titulo=f"Oscilador amortiguado — χ²_ν = {c2_bueno:.2f}, p = {p_bueno:.2f}")
axes[0].plot(t, p[0]*np.exp(-t/p[1]) + p[4], "k--", lw=1, label="envolvente")
axes[0].legend()
plt.show()

print("Resultados:")
lab.reportar(p[1], e[1], "s", nombre="tau")
lab.reportar(p[2], e[2], "rad/s", nombre="omega")
lab.reportar(p[2]/(2*np.pi), e[2]/(2*np.pi), "Hz", nombre="frecuencia")
print()
print(f"factor de calidad Q = ω·τ/2 = {p[2]*p[1]/2:.1f}")

### 7. Ejercicios

1. Ajustá tus datos y reportá $\tau$ y $\omega$ con sus incertezas y con el
   diagnóstico completo ($\chi^2_\nu$, p-valor, residuos).
2. **Prueba de fuego de un modelo:** ajustá usando solo la primera mitad de
   tus datos y predecí la segunda. Graficá la predicción sobre los datos que
   el ajuste no vio. Es la validación más exigente que existe y no lleva más
   de cinco líneas.
3. Compará el $\omega$ del oscilador amortiguado con
   $\omega_0 = \sqrt{k/m}$ usando el $k$ de la Clase 6. Deberían diferir en
   $\omega^2 = \omega_0^2 - 1/\tau^2$. ¿Podés medir esa diferencia, o es más
   chica que tu incerteza?
4. Corré el ajuste con `p0` deliberadamente malas (por ejemplo, $\omega$ el
   doble del correcto) y documentá qué devuelve. Guardá esa figura: es el
   mejor recordatorio de que un ajuste convergido no es un ajuste correcto.

In [ ]:
# Espacio de trabajo para los ejercicios.

### Informe 4 (Clases 9 y 10)

Es el informe de mayor peso del curso. Oscilador simple y amortiguado, con
ajuste no lineal completo y diagnóstico de bondad de ajuste. Se entrega junto
con esta clase la consigna de la Práctica Especial.